In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


def load_data():

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    return X, y, X_test


# -----------------------------
# VIP score calculation
# -----------------------------
def calculate_vip(pls, X, y):

    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_

    p, h = w.shape

    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)

    vip = np.zeros(p)

    for i in range(p):
        weight = np.array([
            (w[i, j] / np.linalg.norm(w[:, j]))**2
            for j in range(h)
        ])
        vip[i] = np.sqrt(p * (s.T @ weight) / total_s)

    return vip


def select_vip_features(X, X_test, y):

    pls = PLSRegression(n_components=20)
    pls.fit(X, y)

    vip_scores = calculate_vip(pls, X, y)

    mask = vip_scores > 1.0

    print("Original wavelengths:", X.shape[1])
    print("Selected wavelengths:", np.sum(mask))

    return X[:, mask], X_test[:, mask]


def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    return X_scaled, X_test_scaled


def cross_validate(X, y):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=1.0,
            l1_ratio=0.7,
            max_iter=100000,
            tol=1e-3,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

    print("CV RMSE:", np.mean(rmse_scores))


def train_and_predict(X, y, X_test):

    model = ElasticNet(
        alpha=1.0,
        l1_ratio=0.7,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X, y)

    preds = model.predict(X_test)

    print("Sample predictions:", preds[:10])

    return preds


def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp06_pls_vip_elasticnet_20260323"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("Submission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = select_vip_features(X, X_test, y)

    X, X_test = scale_features(X, X_test)

    cross_validate(X, y)

    preds = train_and_predict(X, y, X_test)

    save_submission(test, preds)


if __name__ == "__main__":
    main()

Train shape: (1322, 1559)
Test shape: (550, 1558)
Original wavelengths: 1555
Selected wavelengths: 434


/var/folders/01/5r6f25414h3f57k6hs6f66z40000gn/T/ipykernel_2948/498156952.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  vip[i] = np.sqrt(p * (s.T @ weight) / total_s)


CV RMSE: 25.341649744143933
Sample predictions: [193.82202041 166.36589378 153.46078694 147.25686743 138.87481889
 130.17113919 122.99138013 117.37499012 112.91540115 109.36027945]
Submission saved to: ../submissions/exp06_pls_vip_elasticnet_20260323.csv
    0           1
0  95  193.822020
1  96  166.365894
2  97  153.460787
3  98  147.256867
4  99  138.874819
